In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mengukur Qubit

<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan untuk menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Untuk mendapatkan informasi tentang keadaan sebuah Qubit, kamu bisa _mengukurnya_ ke sebuah [classical bit](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Clbit). Di Qiskit, pengukuran dilakukan dalam basis komputasional, yaitu basis Pauli-$Z$ satu-Qubit. Oleh karena itu, pengukuran menghasilkan 0 atau 1, tergantung pada tumpang tindih dengan eigenstate Pauli-$Z$ $|0\rangle$ dan $|1\rangle$:

$$
|q\rangle \xrightarrow{measure}\begin{cases}
      0 (\text{outcome}+1), \text{with probability } p_0=|\langle q|0\rangle|^{2}\text{,} \\
      1 (\text{outcome}-1), \text{with probability } p_1=|\langle q|1\rangle|^{2}\text{.}
    \end{cases}
$$

## Pengukuran di tengah Circuit
Pengukuran di tengah Circuit adalah komponen utama dari dynamic circuits. Sebelum `qiskit-ibm-runtime` v0.43.0, `measure` adalah satu-satunya instruksi pengukuran di Qiskit. Namun, pengukuran di tengah Circuit punya persyaratan tuning yang berbeda dari pengukuran _terminal_ (pengukuran yang terjadi di akhir Circuit). Misalnya, kamu perlu mempertimbangkan durasi instruksi saat menyetel pengukuran di tengah Circuit karena instruksi yang lebih lama menghasilkan Circuit yang lebih berisik. Kamu tidak perlu mempertimbangkan durasi instruksi untuk pengukuran terminal karena tidak ada instruksi setelah pengukuran terminal.

Di `qiskit-ibm-runtime` v0.43.0, instruksi `MidCircuitMeasure` diperkenalkan. Seperti namanya, ini adalah instruksi pengukuran baru yang dioptimalkan untuk pengukuran di tengah Circuit pada QPU IBM&reg;.

> **Note:** Instruksi `MidCircuitMeasure` dipetakan ke instruksi `measure_2` yang dilaporkan di `supported_instructions` Backend. Namun, `measure_2` tidak didukung di semua Backend. Gunakan `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` untuk menemukan Backend yang mendukungnya. Pengukuran baru mungkin ditambahkan di masa mendatang, tapi ini tidak dijamin.

## Menerapkan pengukuran ke Circuit
Ada beberapa cara untuk menerapkan pengukuran ke Circuit:

### Metode `QuantumCircuit.measure`
Gunakan metode [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure) untuk mengukur sebuah [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class).

Contoh:

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5, 5)
qc.x(0)
qc.x(1)
qc.x(4)
qc.measure(
    range(5), range(5)
)  #  Measures all qubits into the corresponding clbit.

In [2]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure(1, 0)  # Measure qubit 1 into the classical bit 0.

### `Measure` class

The Qiskit [Measure](/docs/api/qiskit/circuit#qiskit.circuit.Measure) class measures the specified qubits.

In [3]:
from qiskit.circuit import Measure

qc = QuantumCircuit(3, 1)
qc.x([0, 1])
qc.append(Measure(), [0], [0])  # measure qubit 0 into clbit 0

### `QuantumCircuit.measure_all` method

To measure all qubits into the corresponding classical bits, use the [`measure_all`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) method. By default, this method adds new classical bits in a `ClassicalRegister` to store these measurements.

In [4]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_all()  # Measure all qubits.

### Kelas `Measure`
Kelas Qiskit [Measure](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Measure) mengukur Qubit yang ditentukan.

In [5]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_active()  # Measure qubits that are not idle, that is, qubits 0 and 2.

<span id="midcircuit"></span>
### `MidCircuitMeasure` method


Use `MidCircuitMeasure` to apply a mid-circuit measurement (requires `qiskit-ibm-runtime` v0.43.0 or later).  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [6]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure
from qiskit.circuit import Measure

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


### Metode `QuantumCircuit.measure_all`
Untuk mengukur semua Qubit ke classical bits yang sesuai, gunakan metode [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all). Secara default, metode ini menambahkan classical bits baru di sebuah `ClassicalRegister` untuk menyimpan hasil pengukuran ini.